# Exploratory Analysis – Price Series Sanity Check

**Phase 1.6 – Notebook Sanity Check**
- Uses production module `src.data.stock_data` – no duplicate yfinance calls
- Visual inspection detects flatlines, unadjusted splits, late IPOs
- Plotly line chart: one line per ticker

Run top-to-bottom in Codespace Jupyter kernel.

In [ ]:
from src.data.stock_data import (
    DEFAULT_TICKERS,
    DEFAULT_START_DATE,
    DEFAULT_END_DATE,
    download_price_data,
    clean_price_data,
    fetch_and_cache,
    validate_price_data,
    summary_to_markdown,
)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print(f"Tickers: {DEFAULT_TICKERS}")
print(f"Date range: {DEFAULT_START_DATE} to {DEFAULT_END_DATE or 'today'}")

In [ ]:
# Fetch via production pipeline – cached if exists
tickers = DEFAULT_TICKERS[:5]  # small universe for fast plot
df = fetch_and_cache(tickers, start="2020-01-01", end=None)
print(f"Shape: {df.shape}")
print(df.head())

In [ ]:
# Validation summary
summary = validate_price_data(df)
summary

In [ ]:
# Markdown rendering of summary
from IPython.display import Markdown
Markdown(summary_to_markdown(summary))

In [ ]:
# Plotly line chart – one line per ticker, hover shows ticker
fig = go.Figure()
for col in df.columns:
    fig.add_trace(go.Scatter(x=df.index, y=df[col], mode='lines', name=col, hovertemplate=f'{col}: %{{y:.2f}}<br>%{{x}}<extra></extra>'))

fig.update_layout(
    title="Adjusted Close Price Series – Sanity Check",
    xaxis_title="Date",
    yaxis_title="Adjusted Close (USD)",
    hovermode="x unified",
    template="plotly_white",
    height=500
)
from IPython.display import HTML
HTML(fig.to_html())
# Visual checks:
# - No flatlines for months (stale data)
# - No -50% drops (unadjusted split)
# - All series start at same date (no late IPO)

In [ ]:
# Normalized price (base 100) for relative comparison
df_norm = df / df.iloc[0] * 100
fig2 = px.line(df_norm, title="Normalized Price (Base 100) – Relative Performance")
fig2.update_layout(hovermode="x unified", template="plotly_white", height=500)
from IPython.display import HTML
HTML(fig2.to_html())

## Conclusion
- Data is rectangular, NaN-free, positive
- Trading days/year ~252 – annualization correct
- Plot shows no splits, no flatlines, common window
- Ready for Phase 2 Returns calculation

## Portfolio Optimization Comparison (Phase 4, Task 4.10)

We now compare the four Phase 4 portfolios — equal-weight benchmark,
minimum-variance, maximum-Sharpe, and target-return — on the same
`(μ, Σ)` estimated from the processed price CSV. All metrics are
annualized.

The plotly grouped bar chart shows the weight vector for each strategy
side by side, which makes concentration visible at a glance.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML

from src.data.stock_data import load_prices
from src.preprocessing.returns import process_returns_pipeline
from src.linear_algebra.covariance import compute_covariance_matrix
from src.optimization import compare_portfolios

# Load the same data file the CLI uses
repo_root = Path.cwd().parent
csvs = sorted((repo_root / "data" / "processed").glob("prices_clean_*.csv"))
assert csvs, "Run `python -m src.data.stock_data` first to generate the CSV"
prices = load_prices(str(csvs[-1]))

returns, mu, sigma, _ = process_returns_pipeline(prices)
Sigma = compute_covariance_matrix(returns, annualize=True)

df = compare_portfolios(mu, Sigma, risk_free_rate=0.02)
df

In [ ]:
tickers = list(mu.index)
colors = {"EqualWeight": "#888888",
          "MinVolatility": "#2ca02c",
          "MaxSharpe": "#d62728"}

fig = go.Figure()
for strategy in df.index:
    if strategy not in colors:
        continue
    fig.add_trace(go.Bar(
        name=strategy,
        x=tickers,
        y=df.loc[strategy, tickers].astype(float).values,
        marker_color=colors[strategy],
    ))
fig.update_layout(
    title="Portfolio Weight Vectors by Strategy",
    barmode="group",
    xaxis_title="Ticker",
    yaxis_title="Weight",
    yaxis_range=[0, 1],
)
HTML(fig.to_html())

### Expected observations

- `MinVolatility.volatility` is the smallest volatility in the table (by definition).
- `MaxSharpe.sharpe` is the largest Sharpe in the table (by construction).
- `EqualWeight` gives every ticker the same 1/N allocation.
- `MaxSharpe` typically concentrates on 2–4 assets on real data.
- All weight vectors sum to 1 within 1e-6.

If any of the first three fail on real data, investigate the optimizer
(e.g., max-Sharpe may have found a local optimum — spec Section 8).

## Risk Analysis & Efficient Frontier (Phase 5)

Phase 5 adds four diagnostics on top of the Phase 4 optimizers:

- **VaR (95%, daily):** how bad can a single day be at 95% confidence?
- **CVaR (95%, daily):** given that we are in the tail, what is the average loss?
- **Max drawdown:** the worst peak-to-trough cumulative decline.
- **Efficient frontier:** the complete risk-return landscape, with the
  min-variance, max-Sharpe, and equal-weight portfolios labeled.

All risk metrics are computed on the **historical daily return series**
of each portfolio (fixed weights, daily rebalancing), not on the annualized
`mu` and `Sigma`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML

from src.data.stock_data import load_prices
from src.preprocessing.returns import process_returns_pipeline
from src.linear_algebra.covariance import compute_covariance_matrix
from src.risk import (
    compare_risk_metrics,
    analyze_frontier,
    historical_var,
    historical_cvar,
    maximum_drawdown,
    portfolio_historical_returns,
    drawdown_series,
)
from src.optimization import maximum_sharpe_portfolio
from src.visualization.charts import (
    plot_efficient_frontier,
    plot_drawdown,
    plot_var_histogram,
)

repo_root = Path.cwd().parent
csv = sorted((repo_root / "data" / "processed").glob("prices_clean_*.csv"))[-1]
prices = load_prices(str(csv))
returns, mu, sigma, _ = process_returns_pipeline(prices)
Sigma = compute_covariance_matrix(returns, annualize=True)

risk_table = compare_risk_metrics(mu, Sigma, returns, risk_free_rate=0.02)
risk_table.round(4)

In [ ]:
analysis = analyze_frontier(mu, Sigma, risk_free_rate=0.02, n_points=50)
HTML(plot_efficient_frontier(analysis).to_html())

In [ ]:
# Drawdown and VaR histogram for the Max-Sharpe portfolio
ms = maximum_sharpe_portfolio(mu, Sigma, risk_free_rate=0.02)
R_p_ms = portfolio_historical_returns(ms["weights"], returns)

dd = drawdown_series(R_p_ms)
var_ms = historical_var(R_p_ms, confidence_level=0.95)
cvar_ms = historical_cvar(R_p_ms, confidence_level=0.95)
mdd_ms = maximum_drawdown(R_p_ms)

print(f"Max-Sharpe portfolio — VaR95={var_ms*100:.2f}%  CVaR95={cvar_ms*100:.2f}%  MDD={mdd_ms*100:.2f}%")
HTML(plot_drawdown(dd, portfolio_name="Max Sharpe").to_html())

In [8]:
HTML(plot_var_histogram(R_p_ms, var_ms, portfolio_name="Max Sharpe", confidence_level=0.95).to_html())